# 動画を1本作る（このページの使い方）

このページは **2回に分けて** 押す。コードは読まなくていい。

今このノートが焼くのは、入っているサクラの絵で **15秒を1本**（6秒と9秒をつなぐ）。別の絵や文章に変える欄は無い。

## 1. 最初は CPU（動画はまだ出ない）

右上 **ランタイム → ランタイムのタイプを変更 → CPU**。

上から順に、再生ボタンを押す。

1. **1-1 今日のDrive** … 今日の Google アカウントを選ぶ。フォルダ名はそのままでよい。
2. **1-2 プログラムを取る**
3. **1-3 道具を入れる**
4. **1-4 重みをDriveに置く** … ここが長い（100GB超）。すでにファイルがあれば飛ばす。

## 2. 次に G4（ここで mp4 が出る）

**ランタイムのタイプを G4 GPU に変える。** 機械が変わるので、最初からやり直しに見える。

5. **2-1 今日のDriveをもう一度** … アカウントを選び直す。フォルダ名は 1 と同じ。
6. **2-2 動画を焼く** … 終わるまで待つ。mp4 は今日の Drive の `h3-runner/output` に出る。

一番下の **比較** は押さなくていい。画質の試し打ちで、本編ではない。

Drive のアカウントは毎回違ってよい。前回の中身はこの機械に残っていない。

## 1. CPU（GPUは切る）

ランタイムを **CPU** にする。下の 1-1 から 1-4 まで、上から順に押す。ここでは動画は出ない。

### 1-1 今日のDrive

マウント画面で、今回の Google アカウントを選ぶ。フォルダ名は、そのマイドライブの中。無ければこのセルが作る。

In [ ]:
#@title 1-1 今日のDrive
#@markdown マウントで今日のアカウントを選ぶ。フォルダが無ければ、このセルがマイドライブの中に作る。
WEIGHTS_FOLDER = "h3-weights"  #@param {type:"string"}
OUT_FOLDER = "h3-runner/output"  #@param {type:"string"}

import os
import shutil
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=True)
root = Path("/content/drive/MyDrive")
if not root.is_dir():
    raise SystemExit("マイドライブが見えない。マウントで、今回使うアカウントを選び直す。")
if not WEIGHTS_FOLDER.strip() or not OUT_FOLDER.strip():
    raise SystemExit("WEIGHTS_FOLDER と OUT_FOLDER を、今回のマイドライブの中の名前にする。")
WEIGHTS = root / WEIGHTS_FOLDER
OUT = root / OUT_FOLDER
WEIGHTS.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
os.environ["H3_HF_CACHE"] = str(WEIGHTS)
os.environ["H3_OUT_DIR"] = str(OUT)
os.environ["HF_HOME"] = "/content/tmp_hf/hf-home"

token = ""
try:
    token = userdata.get("HF_TOKEN") or ""
except Exception:
    token = ""
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN をシークレットから読みました（値は表示しません）")
else:
    print("HF_TOKEN は未設定。401 のときだけ、ライセンス同意後にシークレットへ入れてこのセルからやり直す。")

usage = shutil.disk_usage("/content/drive")
disk = shutil.disk_usage("/")
print(f"参考: /content/drive の statvfs 空き {usage.free / 1e9:.1f} GB / 全体 {usage.total / 1e9:.1f} GB（10^9）")
print("Colab のマウントは VM ディスクの値を返すので Drive 実容量ではない。この数字では止めない。")
print(f"ローカル / の空き {disk.free / 1e9:.1f} GB。drivefs のキャッシュはここを使う。")
print("両方置く合計は 210.3GB。シャードを1本ずつ落とす。")
print("今回の重み:", WEIGHTS)
print("今回の完成 mp4:", OUT)

### 1-2 プログラムを取る

焼くプログラムをこの機械に取る。失敗したら、このセルだけもう一度。

In [ ]:
#@title 1-2 プログラムを取る
import subprocess
from pathlib import Path

def clone_runner():
    root = Path("/content/Research")
    branch = "cursor/h3-sfw-colab-6dc5"
    repo = "https://github.com/fireworker011/Research.git"
    script = root / "h3-runner" / "run_h3.py"
    if not script.is_file():
        if root.exists():
            raise SystemExit(f"{root} があるが {script} が無い。このフォルダを消してやり直す。")
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repo, str(root)])
    else:
        subprocess.check_call(["git", "-C", str(root), "fetch", "--depth", "1", "origin", branch])
        subprocess.check_call(["git", "-C", str(root), "checkout", branch])
        subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only", "origin", branch])
    if not script.is_file():
        raise SystemExit(f"ランナーが無い: {script}")
    print("runner", script)
    return script

clone_runner()


### 1-3 道具を入れる

重みを落とすための道具だけ入れる。ここでは動画は焼かない。

In [ ]:
#@title 1-3 道具を入れる
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.25.0"])
print("huggingface_hub ok")


### 1-4 重みをDriveに置く

**ここが長い。** 無ければ1本ずつ、今日の Drive に落とす。すでにあるファイルは飛ばす。GPU は使わない。

終わるまでこのページを閉じない。切れたら、このセルだけもう一度押す。続きから落ちる。

LoRA は、今回の重みフォルダの `loras/` に置く。

置くファイル:

- Turbo: Hugging Face `lightx2v/Minimax-h3-Turbo` の `minimax_h3_fl2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors`
- Motion Continuity Repair V2: Hugging Face `JOKER141/MiniMax-H3-General-Motion-Continuity-Repair` の `Motion_Repair_V2.safetensors`
- Combat: Hugging Face `JOKER141/MiniMax-H3-Combat-Base-V2` の `H3_Combat_V2.safetensors`

In [ ]:
#@title 1-4 重みをDriveに置く（長い）
import os
import subprocess
import sys
import threading
from pathlib import Path

def run_logged(cmd):
    """Colab does not show a child process's own fds. Print them from this cell."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    stderr_lines = []

    def pump(stream, acc):
        for line in stream:
            print(line, end="", flush=True)
            if acc is not None:
                acc.append(line)

    out_thread = threading.Thread(target=pump, args=(proc.stdout, None))
    err_thread = threading.Thread(target=pump, args=(proc.stderr, stderr_lines))
    out_thread.start()
    err_thread.start()
    code = proc.wait()
    out_thread.join()
    err_thread.join()
    if code != 0:
        print(f"\n終了コード {code}。stderr 全文:", flush=True)
        print("".join(stderr_lines), flush=True)
        raise SystemExit(code)
    return code

def clone_runner():
    root = Path("/content/Research")
    branch = "cursor/h3-sfw-colab-6dc5"
    repo = "https://github.com/fireworker011/Research.git"
    script = root / "h3-runner" / "run_h3.py"
    if not script.is_file():
        if root.exists():
            raise SystemExit(f"{root} があるが {script} が無い。このフォルダを消してやり直す。")
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repo, str(root)])
    else:
        subprocess.check_call(["git", "-C", str(root), "fetch", "--depth", "1", "origin", branch])
        subprocess.check_call(["git", "-C", str(root), "checkout", branch])
        subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only", "origin", branch])
    if not script.is_file():
        raise SystemExit(f"ランナーが無い: {script}")
    print("runner", script)
    return script

script = clone_runner()
cmd = [
    sys.executable, "-u", str(script),
    "--preset", "orbis01",
    "--cache-dir", os.environ["H3_HF_CACHE"],
    "--out-dir", os.environ["H3_OUT_DIR"],
    "--prepare-weights",
    "--vram-gb", "95",
    "--host-ram-gb", "176.9",
]
run_logged(cmd)
import sys

sys.path.insert(0, "/content/Research/h3-runner")
from h3_runner.loras import prepare_fast_loras

lora_dir = Path(os.environ["H3_HF_CACHE"]) / "loras"
prepare_fast_loras(lora_dir)
print("LoRA:", lora_dir)
print("重み準備が終わった。ランタイムを G4 に変えて B へ進む。")


## 2. G4（ここで動画が出る）

ランタイムのタイプを **G4 GPU** に変える。CPU のときとは別の機械になる。Drive はもう一度選ぶ。

### 2-1 準備

今日のアカウントを選び直す。フォルダ名は 1-1 と同じにする。重みフォルダが無いと、ここで止まってフォルダ名を出す。

In [ ]:
#@title 2-1 今日のDriveをもう一度
#@markdown A で使ったのと同じアカウント、同じフォルダ名にする。アカウントを変えたら、その Drive に重みがある方を書く。
WEIGHTS_FOLDER = "h3-weights"  #@param {type:"string"}
OUT_FOLDER = "h3-runner/output"  #@param {type:"string"}

import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=True)
root = Path("/content/drive/MyDrive")
if not root.is_dir():
    raise SystemExit("マイドライブが見えない。マウントで、今回使うアカウントを選び直す。")
WEIGHTS = root / WEIGHTS_FOLDER
OUT = root / OUT_FOLDER
if not WEIGHTS.is_dir():
    names = sorted(p.name for p in root.iterdir() if p.is_dir())[:30]
    raise SystemExit(
        f"今回の Drive に重みが無い: {WEIGHTS}\n"
        "WEIGHTS_FOLDER を、このアカウントにあるフォルダ名に変えるか、CPU の A をこのアカウントで先に実行する。\n"
        "見えているフォルダ: " + ", ".join(names)
    )
OUT.mkdir(parents=True, exist_ok=True)
os.environ["H3_HF_CACHE"] = str(WEIGHTS)
os.environ["H3_OUT_DIR"] = str(OUT)
os.environ["HF_HOME"] = str(WEIGHTS / "hf-home")
print("今回の重み:", WEIGHTS)
print("今回の完成 mp4:", OUT)

token = ""
try:
    token = userdata.get("HF_TOKEN") or ""
except Exception:
    token = ""
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN をシークレットから読みました（値は表示しません）")

usage = shutil.disk_usage("/content/drive")
print(f"Drive 空き {usage.free / 1e9:.1f} GB / 全体 {usage.total / 1e9:.1f} GB（10^9）")

import torch
if not torch.cuda.is_available():
    raise SystemExit("GPU がオフです。ランタイムを G4 にして、このセルからやり直す。")
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024 ** 3
mem = Path("/proc/meminfo").read_text(encoding="utf-8")
host = int(next(line.split()[1] for line in mem.splitlines() if line.startswith("MemTotal:"))) / 1024 ** 2
print(f"GPU: {props.name}  VRAM: {vram:.1f} GB  ホストRAM: {host:.1f} GB  torch {torch.__version__}")
os.environ["H3_VRAM_GB"] = f"{vram:.4f}"
os.environ["H3_HOST_RAM_GB"] = f"{host:.4f}"
if vram < 24:
    raise SystemExit("VRAM が 24GB 未満です。G4 に変えてやり直す。")

if shutil.which("ffmpeg") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg"])

pkgs = [
    "git+https://github.com/huggingface/diffusers.git@5ff8e59ff9fe81c6e2df4fb4c6ea0d97a5df5ab2",
    "transformers>=4.45.0",
    "accelerate>=0.34.0",
    "safetensors>=0.4.3",
    "huggingface_hub>=0.25.0",
    "av>=11.0.0",
    "imageio>=2.34.0",
    "imageio-ffmpeg>=0.5.0",
    "soundfile>=0.12.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *pkgs])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "torchao==0.18.0"])
from torchao.quantization import FqnToConfig
print("torchao FqnToConfig ok", "ffmpeg", shutil.which("ffmpeg"))

def clone_runner():
    root = Path("/content/Research")
    branch = "cursor/h3-sfw-colab-6dc5"
    repo = "https://github.com/fireworker011/Research.git"
    script = root / "h3-runner" / "run_h3.py"
    if not script.is_file():
        if root.exists():
            raise SystemExit(f"{root} があるが {script} が無い。このフォルダを消してやり直す。")
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repo, str(root)])
    else:
        subprocess.check_call(["git", "-C", str(root), "fetch", "--depth", "1", "origin", branch])
        subprocess.check_call(["git", "-C", str(root), "checkout", branch])
        subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only", "origin", branch])
    if not script.is_file():
        raise SystemExit(f"ランナーが無い: {script}")
    print("runner", script)
    return script

clone_runner()

### 2-2 動画を焼く

このセルが mp4 を出す。重みのダウンロードはしない。今日の Drive を読む。

サクラの絵で 6秒 + 9秒を焼き、15秒の1本につなぐ。同じファイルが Drive にあれば、その本は飛ばす。

切れたら、画面が生きていればこのセルだけ。死んでいれば 2-1 からやり直す。

In [ ]:
#@title 2-2 動画を焼く
import os
import subprocess
import sys
import threading
from pathlib import Path

def run_logged(cmd):
    """Colab does not show a child process's own fds. Print them from this cell."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    stderr_lines = []

    def pump(stream, acc):
        for line in stream:
            print(line, end="", flush=True)
            if acc is not None:
                acc.append(line)

    out_thread = threading.Thread(target=pump, args=(proc.stdout, None))
    err_thread = threading.Thread(target=pump, args=(proc.stderr, stderr_lines))
    out_thread.start()
    err_thread.start()
    code = proc.wait()
    out_thread.join()
    err_thread.join()
    if code != 0:
        print(f"\n終了コード {code}。stderr 全文:", flush=True)
        print("".join(stderr_lines), flush=True)
        raise SystemExit(code)
    return code

script = Path("/content/Research/h3-runner/run_h3.py")
if not script.is_file():
    raise SystemExit(f"ランナーが無い: {script}。B. 準備からやり直す。")
cmd = [
    sys.executable, "-u", str(script),
    "--preset", "orbis01",
    "--cache-dir", os.environ["H3_HF_CACHE"],
    "--out-dir", os.environ["H3_OUT_DIR"],
    "--vram-gb", os.environ["H3_VRAM_GB"],
    "--host-ram-gb", os.environ["H3_HOST_RAM_GB"],
    "--seed", "0",
]

lora = Path(os.environ["H3_HF_CACHE"]) / "loras" / "minimax_h3_fl2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors"
if not lora.is_file():
    raise SystemExit(f"Turbo LoRA が無い: {lora}。CPU の「A. 重み準備」を先に実行する。")
cmd.extend(["--steps", "9", "--video-shift", "6", "--lora", f"{lora}:1.0"])
run_logged(cmd)
out = Path(os.environ["H3_OUT_DIR"]) / "orbis01.mp4"
if not out.is_file():
    raise SystemExit(f"完成 mp4 がありません: {out}")
print("完成:", out, "bytes", out.stat().st_size)


### 押さなくていい（画質の比較）

本編の15秒とは別の試し打ち。画質を見たいときだけ、2-2 のあとに押す。

同じ6秒を、Turbo だけ、Turbo+Combat、Turbo+Combat+Repair、Turbo+Repair で出す。保存先は 2-1 で指定したフォルダ。

In [ ]:
#@title 押さなくていい（画質の比較）
import os
import re
import subprocess
import sys
import threading
from pathlib import Path

def run_capture(cmd):
    """Print the child live, and keep the text so one failure can continue."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    chunks = []

    def pump(stream):
        for line in stream:
            print(line, end="", flush=True)
            chunks.append(line)

    out_thread = threading.Thread(target=pump, args=(proc.stdout,))
    err_thread = threading.Thread(target=pump, args=(proc.stderr,))
    out_thread.start()
    err_thread.start()
    code = proc.wait()
    out_thread.join()
    err_thread.join()
    return code, "".join(chunks)

script = Path("/content/Research/h3-runner/run_h3.py")
prompt = Path("/content/Research/h3-runner/prompts/orbis01_t2va_6s.txt")
still = Path("/content/Research/h3-runner/assets/sakura-ref.jpg")
if not script.is_file():
    raise SystemExit(f"ランナーが無い: {script}。B. 準備からやり直す。")
lora_dir = Path(os.environ["H3_HF_CACHE"]) / "loras"
turbo = lora_dir / "minimax_h3_fl2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors"
combat = lora_dir / "H3_Combat_V2.safetensors"
repair = lora_dir / "Motion_Repair_V2.safetensors"
out_dir = Path(os.environ["H3_OUT_DIR"])
baseline = out_dir / "orbis01_6s.mp4"
print("A 基準:", baseline, "bytes", baseline.stat().st_size if baseline.is_file() else "無い")
print("プロンプト:", prompt)
print("静止画:", still)
print("Combat は BUNNY（作者 FourBunny）。Repair は BUNNY Motion Repair V2（配布 JOKER141）。動画にクレジットを書く。")

variants = [
    ("B", [f"{turbo}:1.0"], out_dir / "test_b_fl2va_turbo.mp4"),
]
if combat.is_file():
    variants.append(("C", [f"{turbo}:1.0", f"{combat}:0.7"], out_dir / "test_c_turbo_combat.mp4"))
    variants.append(("D", [f"{turbo}:1.0", f"{combat}:0.7", f"{repair}:0.6"], out_dir / "test_d_turbo_combat_repair.mp4"))
else:
    print("H3_Combat_V2.safetensors が無いので C と D は飛ばす。A. 重み準備からやり直す。")
variants.append(("D'", [f"{turbo}:1.0", f"{repair}:0.6"], out_dir / "test_dp_turbo_repair.mp4"))
rows = {"A": {"load": "—", "text": "—", "denoise": "—", "vae": "—", "out": baseline.name if baseline.is_file() else "無い"}}
for name, loras, dest in variants:
    missing = [Path(item.rsplit(":", 1)[0]) for item in loras if not Path(item.rsplit(":", 1)[0]).is_file()]
    if missing:
        rows[name] = {"load": "失敗", "text": "失敗", "denoise": "失敗", "vae": "失敗", "out": "LoRA が無い: " + ", ".join(path.name for path in missing)}
        print(f"{name} を飛ばす。LoRA が無い:", ", ".join(str(path) for path in missing))
        continue
    cmd = [
        sys.executable, "-u", str(script),
        "--task", "fl2va",
        "--prompt-file", str(prompt),
        "--image", str(still),
        "--duration", "6",
        "--width", "768",
        "--height", "1344",
        "--seed", "0",
        "--steps", "9",
        "--video-shift", "6",
        "--offload", "bf16",
        "--cache-dir", os.environ["H3_HF_CACHE"],
        "--vram-gb", os.environ.get("H3_VRAM_GB", "95"),
        "--host-ram-gb", os.environ.get("H3_HOST_RAM_GB", "176.9"),
        "--out", str(dest),
    ]
    for spec in loras:
        cmd.extend(["--lora", spec])
    print(f"\n==== {name} → {dest.name} ====", flush=True)
    code, text = run_capture(cmd)
    found = re.search(r"TIMING load=([0-9.]+) text=([0-9.]+) denoise=([0-9.]+) vae=([0-9.]+)", text)
    if code == 0 and found and dest.is_file():
        rows[name] = {
            "load": found.group(1),
            "text": found.group(2),
            "denoise": found.group(3),
            "vae": found.group(4),
            "out": f"{dest.name} ({dest.stat().st_size} bytes)",
        }
    else:
        tail = text.strip().splitlines()[-3:]
        rows[name] = {
            "load": "失敗",
            "text": "失敗",
            "denoise": "失敗",
            "vae": "失敗",
            "out": f"終了コード {code}: " + " / ".join(tail),
        }
        print(f"{name} は落ちた。次へ進む。", flush=True)

skipped = {"load": "飛ばした", "text": "飛ばした", "denoise": "飛ばした", "vae": "飛ばした", "out": "Combat が無い"}
rows.setdefault("C", skipped)
rows.setdefault("D", skipped)
print("\n| 段 | A 基準 | B Turbo | C +Combat 0.7 | D C+Repair 0.6 | D' B+Repair 0.6 |")
print("|---|---|---|---|---|---|")
labels = [("読み込み", "load"), ("テキストエンコード", "text"), ("ノイズ除去", "denoise"), ("VAE", "vae"), ("出力", "out")]
order = ["A", "B", "C", "D", "D'"]
for label, key in labels:
    print("| " + " | ".join([label] + [rows[name][key] for name in order]) + " |")
failed = [name for name in ("B", "C", "D", "D'") if rows[name]["load"] == "失敗"]
if failed:
    raise SystemExit("落ちた段: " + ", ".join(failed) + "。表は上。")
